<p align='center'><img src='https://dsevilla.github.io/tcdm-public/img/tcdm-lumon.png' alt='TCDM — Tecnologías de Computación de Datos Masivos' width='600'></p>


# Sesión 1 — Entorno Hadoop con contenedores

En esta primera sesión se pone en marcha el clúster Hadoop que se empleará
durante todo el curso. El clúster ya está construido y publicado como
imágenes de contenedor: aquí no se instala Hadoop, se **arranca, se entiende
y se comprueba** un entorno distribuido pequeño con HDFS, YARN y MapReduce.

Este notebook mezcla explicación conceptual (celdas markdown) con órdenes que
se ejecutan de verdad contra el clúster (celdas de código). No es necesario
memorizar las órdenes; sí es importante entender **en qué máquina** se
ejecuta cada una, **con qué usuario** y **sobre qué sistema de ficheros**
actúa, porque esa distinción se repetirá en toda la asignatura.

> **Cómo se evalúa el trabajo de esta sesión.** Cada sesión práctica termina
> con una sección «Evidencias para la siguiente sesión» que indica qué
> preparar. En una sesión posterior tendrás una reunión individual breve
> (unos minutos) con el profesor en la que deberás **mostrar en vivo, en tu
> propio ordenador**, las evidencias indicadas allí (el clúster en marcha,
> las órdenes ejecutadas, las interfaces web) y **entregar una memoria breve
> (una o dos páginas)** que explique lo realizado. Este esquema —evidencias
> en pantalla más memoria escrita, revisadas en una reunión breve— se repite
> en el resto de sesiones del curso.


### Requisitos previos en Windows: WSL2 y Docker Desktop

Esta sesión ejecuta el kernel de Jupyter en tu propio equipo — a partir de S2
el kernel se ejecutará dentro del propio clúster, pero aquí todavía no existe
clúster que lo aloje. Si usas Windows, `!docker` **no funciona solo con
WSL2 instalado**: hace falta además Docker Desktop, correctamente
configurado. Sigue estos pasos antes de empezar:

1. **Instala WSL2.** Abre PowerShell como administrador y ejecuta:
   ```powershell
   wsl --install
   ```
   Esto instala WSL2 y una distribución Ubuntu por defecto. Reinicia el
   equipo si te lo pide.

2. **Instala Docker Desktop para Windows** desde
   [docker.com](https://www.docker.com/products/docker-desktop/) y, durante
   o después de la instalación, comprueba en Settings → General que
   **"Use the WSL 2 based engine"** está activado.

3. **Activa la integración WSL para tu distribución.** En Docker Desktop, ve
   a Settings → Resources → WSL Integration y activa el interruptor junto al
   nombre de tu distribución (por ejemplo, `Ubuntu`). **Este paso es el que
   más se olvida**: sin él, `docker` no se encuentra dentro de WSL2 aunque
   Docker Desktop esté instalado y en ejecución.

4. **Arranca Docker Desktop** y espera a que quede en marcha (no basta con
   tenerlo instalado; el motor debe estar realmente arrancado).

5. **Instala Visual Studio Code** con las extensiones **WSL**, **Python** y
   **Jupyter**. Abre el proyecto *desde dentro* de WSL2 —por ejemplo,
   ejecutando `code .` desde una terminal de Ubuntu, o con "Connect to WSL"
   desde la paleta de comandos—, no como una carpeta nativa de Windows. Es
   preferible clonar el repositorio dentro del sistema de ficheros de WSL2
   (p. ej. `~/tcdm`) en lugar de `/mnt/c/...`: además de evitar problemas de
   integración, el rendimiento de E/S es mucho mejor.

6. **Selecciona como kernel del notebook un intérprete de Python de WSL2**
   (no uno de Windows), con `ipykernel` instalado.

7. **Comprueba antes de continuar**, en una terminal de WSL2:
   ```bash
   docker --version
   docker compose version
   docker info
   ```
   Si algún comando falla, revisa los pasos 2–4 antes de seguir: el resto
   del notebook asume que estas órdenes ya funcionan desde la propia
   terminal de WSL2.

> **En Linux y macOS** no es necesario nada de lo anterior: basta con tener
> Docker (Docker Engine en Linux, o Docker Desktop en macOS) instalado y en
> ejecución.

### Diapositivas de la sesión

Las diapositivas se generan con el paquete [`jupyter-notebook-slide`](https://github.com/dsevilla/jupyter-notebook-slide).
El alias `%%diapositiva` permite mantener en español los tipos usados en la sesión y produce la misma salida HTML en Jupyter, Colab y la referencia HTML publicada.

En cada sesión encontrarás varios tipos de diapositivas que te ayudarán a seguir el hilo:

- **Sección** (`titulo`): abre la sesión y resume qué vamos a ver y qué haremos.
- **A continuación** (`avance`): anuncia el contenido de la parte siguiente, para saber en qué punto del programa estamos.
- **Recapitulación** (`resumen`): condensa la parte anterior en puntos clave para recordarlos y revisarlos después.
- **Pregunta guía** (`pregunta`): plantea cuestiones que orientan lo que viene a continuación; conviene intentar responderlas antes de seguir.
- **Evaluación de la sesión** (`evaluacion`): recuerda cómo se evalúa el trabajo de esta sesión.


In [ ]:
%pip install -q "jupyter-notebook-slide @ git+https://github.com/dsevilla/jupyter-notebook-slide.git"
%load_ext notebook_slide

import notebook_slide as jnbs

# Colores para TCDM.
jnbs.configure(
    background="#eaf2f8",
    foreground="#1f2933",
    border="#c9d6e1",
    heading="#0c304d",
    subheading="#0c304d",
    link="#174f7a",
    code_background="#eaf2f8",
    code_foreground="#0c304d",
    quote_background="#ffffff",
    font_family="Atkinson Hyperlegible, Inter, Aptos, Segoe UI, Helvetica, Arial, sans-serif",
    font_url="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&display=swap",
)
jnbs.register_slide_type("avance", "A continuación", "#174f7a")
jnbs.register_slide_type("resumen", "Recapitulación", "#a54467")
jnbs.register_slide_type("pregunta", "Pregunta guía", "#0c304d")
jnbs.register_slide_type("evaluacion", "Evaluación de la sesión", "#b3701a")
jnbs.register_slide_type(
    "titulo",
    "Sección",
    "#174f7a",
    layout="title",
    background="linear-gradient(135deg, #0c304d, #174f7a 62%, #a54467)",
    foreground="#ffffff",
    border="transparent",
    heading="#ffffff",
    subheading="#dceaf4",
)
jnbs.register_alias("diapositiva")

In [ ]:
%%diapositiva titulo
# Sesión 1: HDFS, YARN y MapReduce
- Por qué existe Big Data y qué problema resuelve MapReduce
- HDFS, YARN y MapReduce: qué papel tiene cada pieza
- Arrancamos un clúster real de cuatro máquinas simuladas con contenedores
- Comprobamos HDFS y YARN con datos y trabajos reales
- Ejecutamos WordCount y vemos dónde se calculó cada parte

In [ ]:
%%diapositiva avance
# A continuación: los conceptos antes del clúster
- Qué es Hadoop y qué hacen HDFS, YARN y MapReduce, y cómo encajan entre sí
- Por qué existe Big Data y por qué Google publicó GFS y MapReduce
- Los objetivos concretos que debes poder demostrar al final de la sesión

In [ ]:
%%diapositiva evaluacion
# Cómo se evalúa esta sesión
- Cada sesión termina con «Evidencias para la siguiente sesión»
- Reunión individual breve con el profesor en una sesión posterior
- Se muestra en vivo, en tu ordenador, lo que pide esa sección
- Se entrega también una memoria breve (una o dos páginas)


## Hadoop, HDFS, YARN y MapReduce

[**Apache Hadoop**](https://hadoop.apache.org/) es una plataforma de software libre para almacenar y procesar
grandes conjuntos de datos mediante un clúster de ordenadores. Su diseño
parte de una idea sencilla: cuando el volumen de información deja de caber o
de procesarse cómodamente en una sola máquina, se reparten los datos y el
trabajo entre varios nodos. El sistema debe coordinar esos nodos, aprovechar
sus recursos y seguir funcionando cuando alguno deja de estar disponible.

En esta sesión intervienen tres piezas principales:

- **[HDFS](https://hadoop.apache.org/docs/current/hadoop-project-dist/hadoop-hdfs/HdfsDesign.html)** proporciona almacenamiento distribuido;
- **[YARN](https://hadoop.apache.org/docs/current/hadoop-yarn/hadoop-yarn-site/YARN.html)** negocia y asigna los recursos de cómputo del clúster;
- **[MapReduce](https://hadoop.apache.org/docs/current/hadoop-mapreduce-client/hadoop-mapreduce-client-core/MapReduceTutorial.html)** proporciona un modelo y un motor de procesamiento distribuido
  que se ejecuta utilizando los recursos concedidos por YARN.

Hadoop puede acceder también a otros sistemas de almacenamiento, como el
sistema local o almacenes de objetos compatibles con S3. En esta primera
parte del curso se utiliza HDFS porque permite estudiar de forma explícita la
división en bloques, la replicación y la relación entre almacenamiento y
procesamiento. HDFS alojará tanto un *data lake* de ficheros Parquet como un
*data warehouse* gobernado por un catálogo. Más adelante se repetirá parte
del diseño sobre almacenamiento de objetos (sesión 3, S3) para comparar
ambos soportes.

### HDFS como sistema de ficheros distribuido

HDFS, *Hadoop Distributed File System*, está diseñado para almacenar
ficheros grandes en un conjunto de máquinas relativamente convencionales. Un
fichero se divide en bloques y esos bloques se distribuyen entre los
DataNodes. El cliente sigue viendo un único nombre de fichero: la división y
la localización física de cada bloque quedan ocultas tras la interfaz de
HDFS.

El sistema favorece el acceso secuencial con un ancho de banda elevado. No
intenta comportarse como un disco local de baja latencia:

- funciona bien con ficheros grandes y lecturas completas o extensas;
- permite que varios nodos lean bloques diferentes en paralelo;
- obtiene tolerancia a fallos manteniendo réplicas de los bloques;
- tiene una latencia mayor que un sistema de ficheros local;
- es poco eficiente cuando se almacenan cantidades enormes de ficheros muy
  pequeños, porque cada uno requiere metadatos en el NameNode;
- está orientado a escribir una vez y leer muchas veces;
- no ofrece el mismo modelo de actualización aleatoria ni de múltiples
  escritores concurrentes que un sistema de ficheros POSIX.

Los conceptos esenciales de HDFS son:

- **NameNode**: mantiene el espacio de nombres, los permisos y la relación
  entre ficheros, bloques y DataNodes. Coordina el acceso, pero no guarda el
  contenido ordinario de todos los ficheros de usuario.
- **DataNode**: almacena bloques en su sistema de ficheros local y atiende
  las lecturas y escrituras solicitadas por los clientes.
- **Bloque HDFS**: unidad lógica en la que se divide un fichero. En este
  laboratorio se utilizan bloques de 64 MiB para que sea más fácil observar
  su distribución con datos de tamaño moderado.
- **Replicación**: mantenimiento de varias copias de un bloque en DataNodes
  diferentes. El laboratorio solicita tres réplicas porque dispone de tres
  DataNodes.
- **Espacio de nombres**: jerarquía de directorios y ficheros que presenta
  HDFS. Se parece visualmente a un sistema Unix, pero no es la jerarquía
  local de ninguno de los contenedores.

Una instalación de producción necesita además una estrategia de alta
disponibilidad para el NameNode (NameNodes activo/*standby*, JournalNodes,
etc.). El laboratorio local utiliza un único NameNode y no simula esa parte
de una instalación de producción.

### YARN como gestor de recursos

YARN, *Yet Another Resource Negotiator*, separa la gestión de los recursos
del clúster del motor que procesa los datos. Gracias a esa separación,
distintas aplicaciones (MapReduce, y más adelante Spark) pueden solicitar CPU
y memoria sin que cada una implemente su propio planificador global.

- el **ResourceManager** mantiene la visión global de los recursos y decide
  dónde se pueden asignar contenedores de ejecución;
- cada **NodeManager** anuncia la CPU y la memoria disponibles en su nodo,
  inicia los contenedores que le asigna YARN y supervisa su ejecución;
- cada aplicación utiliza un **ApplicationMaster**, que negocia recursos para
  sus tareas y sigue su progreso.

Un contenedor YARN **no** es necesariamente un contenedor Docker: en la
terminología de YARN es una asignación de recursos (memoria y vcores) en un
NodeManager. En este laboratorio esos contenedores YARN se ejecutan dentro de
los contenedores Docker que representan los nodos del clúster.

### MapReduce como modelo de procesamiento

MapReduce organiza un cálculo distribuido alrededor de dos tipos de función.
Las tareas **map** leen fragmentos de la entrada y producen pares
clave-valor intermedios. Hadoop agrupa y transfiere esos pares durante la
fase de *shuffle*. Las tareas **reduce** reciben los valores asociados a cada
clave y generan el resultado final.

MapReduce no es el único motor que puede trabajar sobre Hadoop: Spark
también lee HDFS y solicita recursos a YARN, y se usará más adelante en la
asignatura (sesión 4). WordCount se ejecuta aquí con MapReduce porque
permite observar con pocas órdenes todo el recorrido de una aplicación
distribuida:

1. `luser` copia varios ficheros de entrada a HDFS;
2. el NameNode registra sus nombres, bloques y ubicaciones;
3. los DataNodes almacenan las réplicas de esos bloques;
4. el cliente envía WordCount al ResourceManager;
5. el ApplicationMaster solicita contenedores a YARN;
6. los NodeManagers ejecutan las tareas map y reduce;
7. el resultado vuelve a escribirse en HDFS.

Seguir este flujo permite distinguir entre **dónde están los datos**
(responsabilidad de HDFS) y **dónde se ejecutan las tareas** (decisión de
YARN).

### Referencias

Para profundizar en estos temas, se recomienda:

- **White, Tom** (2015). *Hadoop: The Definitive Guide* (4ª edición). O'Reilly Media. — La referencia clásica y completa sobre Hadoop, HDFS, YARN y MapReduce.
- **Miner, Donald & Shook, Adam** (2018). *MapReduce Design Patterns* (2ª edición). O'Reilly Media. — Patrones y mejores prácticas para escribir aplicaciones MapReduce eficientes.
- **Lin, Jimmy & Dyer, Chris** (2010). *Data-Intensive Text Processing with MapReduce*. Morgan & Claypool. — Enfoque práctico en procesamiento de texto a gran escala.

## Por qué existe Big Data y por qué apareció MapReduce

> **Big Data** (Wikipedia): se refiere al estudio y a las aplicaciones de
> conjuntos de datos tan grandes y complejos que las aplicaciones
> tradicionales de procesamiento de datos resultan inadecuadas para
> tratarlos.

La definición señala tres retos que siguen vigentes: el **volumen** de datos
que hay que almacenar, la **velocidad** con la que se generan y se necesitan
procesar, y la **variedad** de formatos —texto, sensores, registros de
aplicación, vídeo— que conviven en un mismo sistema. Ninguno de los tres se
resuelve comprando un ordenador más grande: en algún momento un único disco,
una única memoria o un único procesador dejan de ser suficientes, y hay que
repartir los datos y el trabajo entre varias máquinas.

Ese cambio de escala llegó con una motivación muy concreta. En 2003 y 2004,
Google publicó dos artículos —*The Google File System* (Ghemawat, Gobioff y
Leung) y *MapReduce: Simplified Data Processing on Large Clusters* (Dean y
Ghemawat)— que describían cómo almacenar y procesar datos sobre miles de
ordenadores convencionales, asumiendo que algunos de ellos fallarían mientras
el trabajo estuviera en marcha. Hacia 2005, Doug Cutting y Mike Cafarella se
basaron en esas ideas para crear Apache Hadoop, el proyecto de código abierto
que reimplementa GFS como **HDFS** y ese modelo de programación como
**MapReduce**. YARN se añadió después, en Hadoop 2, para separar la gestión
de recursos del clúster del motor de procesamiento concreto que los utiliza;
por eso hoy MapReduce es una de varias aplicaciones que pueden pedir recursos
a YARN, y Spark —que se usará en una sesión posterior— es otra.

Esa historia explica varias decisiones de diseño que se van a comprobar en
esta misma sesión: por qué HDFS divide los ficheros en bloques grandes y
los replica en vez de confiar en un único disco, por qué el cómputo se envía
a donde están los datos en lugar de mover los datos al cómputo, y por qué el
sistema está pensado para seguir funcionando —no para bloquearse— cuando un
nodo deja de responder.

## Retos tecnológicos de Big Data

La gestión de Big Data requiere abordar múltiples desafíos tecnológicos en diferentes áreas:

### Búsqueda y refinamiento

Para localizar y limpiar datos de múltiples fuentes dispersas y heterogéneas:

- [Lucene](http://lucene.apache.org/)
- [Solr](http://lucene.apache.org/solr/)
- [Elasticsearch](https://www.elastic.co/products/elasticsearch)
- [OpenSearch](https://opensearch.org/) — fork abierto de Elasticsearch tras el cambio de licencia
- [Meilisearch](https://www.meilisearch.com/) — motor de búsqueda ligero y moderno
- [OpenRefine](http://openrefine.org/)

### Serialización

Transformar datos entre distintos sistemas y formatos de almacenamiento:

- [JSON](http://www.json.org/)
- [BSON](http://bsonspec.org/)
- [Apache Avro](http://avro.apache.org/)
- [Google Protocol Buffers](https://developers.google.com/protocol-buffers/)
- [Apache Arrow](https://arrow.apache.org/) — formato columnar en memoria, base de Pandas, Polars y DuckDB
- [gRPC](https://grpc.io/) — RPC moderno construido sobre Protocol Buffers

### Sistemas de almacenamiento

Distribuir el almacenamiento entre múltiples servidores para máximo rendimiento:

- [HDFS](http://hadoop.apache.org/docs/current/hadoop-project-dist/hadoop-hdfs/HdfsUserGuide.html)
- [Amazon S3](http://aws.amazon.com/es/s3/)
- [MinIO](https://min.io/) — almacenamiento de objetos compatible con S3, autoalojable
- [RustFS](https://rustfs.com/) — almacenamiento de objetos compatible con S3 escrito en Rust; es el que usa el laboratorio en la sesión 3
- [Google Cloud Storage](https://cloud.google.com/storage)
- [Azure Data Lake Storage](https://azure.microsoft.com/es-es/products/storage/data-lake-storage/)

### Servidores

Infraestructura en la nube para escalar recursos según demanda:

- [Amazon EC2](http://aws.amazon.com/es/ec2/)
- [Google Cloud Platform](https://cloud.google.com/)
- [Azure](http://azure.microsoft.com/)
- [OpenShift](https://openshift.redhat.com/app/)
- [Tanzu](https://tanzu.vmware.com/)
- [Kubernetes](https://kubernetes.io/) — orquestación de contenedores, estándar de facto para desplegar clústeres
- [Docker](https://www.docker.com/) — la tecnología de contenedores usada en este propio laboratorio

### Procesamiento

Plataformas y herramientas para procesar datos distribuidos:

- [Hadoop](http://hadoop.apache.org/)
- [Hive](http://hive.apache.org/)
- [Spark](https://spark.apache.org/)
- [Flink](https://flink.apache.org/)
- [Kafka](https://kafka.apache.org/) — la plataforma de *streaming* de eventos más extendida
- [Airflow](https://airflow.apache.org/)
- [Dagster](https://dagster.io/) — orquestador de datos moderno, alternativa a Airflow
- [dbt](https://www.getdbt.com/) — transformación de datos como código dentro del almacén analítico
- [Beam](https://beam.apache.org/)
- [Dask](https://dask.org/)
- [Trino](https://trino.io/) — motor de consultas SQL distribuido sobre múltiples fuentes
- [DuckDB](https://duckdb.org/) — motor analítico embebido, muy popular para análisis local
- [Polars](https://pola.rs/) — DataFrames de alto rendimiento, alternativa moderna a pandas
- [Kinesis](https://aws.amazon.com/es/kinesis/)

### Bases de datos

Almacenamiento de datos estructurados y semiestructurados a gran escala:

- [HBase](http://hbase.apache.org/)
- [Cassandra](http://cassandra.apache.org/)
- [MongoDB](http://www.mongodb.org/)
- [Amazon DynamoDB](http://aws.amazon.com/es/dynamodb/)
- [Google BigTable](http://research.google.com/archive/bigtable.html)
- [Parquet](https://parquet.apache.org/)
- [Apache Iceberg](https://iceberg.apache.org/) — formato de tabla abierto sobre almacenamiento de objetos
- [Delta Lake](https://delta.io/) — formato de tabla con transacciones ACID, alternativa a Iceberg
- [ClickHouse](https://clickhouse.com/) — base de datos analítica columnar de muy alto rendimiento
- [Snowflake](https://www.snowflake.com/) — almacén de datos en la nube totalmente gestionado
- [BigQuery](https://cloud.google.com/bigquery) — almacén de datos serverless de Google Cloud

### Análisis y BI

Análisis estadístico y visualización de datos:

- [R](http://www.r-project.org/)
- [Greenplum](https://pivotal.io/pivotal-greenplum)
- [Splunk](http://www.splunk.com/)
- [Tableau](http://www.tableausoftware.com/)
- [PowerBI](https://powerbi.microsoft.com)
- [Apache Superset](https://superset.apache.org/) — BI de código abierto
- [Metabase](https://www.metabase.com/) — BI de código abierto sencillo de desplegar

### Procesamiento de lenguaje natural

Análisis de contenidos textuales generados por usuarios:

- [Natural Language Toolkit](http://nltk.org/)
- [spaCy](https://spacy.io/) — librería de NLP de producción, más rápida que NLTK
- [Hugging Face Transformers](https://huggingface.co/docs/transformers) — modelos de lenguaje preentrenados y *pipelines* de NLP

### Machine Learning y Deep Learning

Automatización de decisiones basadas en datos:

- [Weka](http://www.cs.waikato.ac.nz/ml/weka/)
- [SciKit-learn](http://scikit-learn.org/stable/)
- [PyTorch](https://pytorch.org/)
- [TensorFlow](https://www.tensorflow.org/)
- [Keras](https://keras.io/)
- [Hugging Face](https://huggingface.co/) — repositorio y herramientas para modelos preentrenados
- [XGBoost](https://xgboost.ai/) — *gradient boosting* de referencia para datos tabulares
- [LightGBM](https://lightgbm.readthedocs.io/) — *gradient boosting* rápido, alternativa a XGBoost
- [JAX](https://jax.readthedocs.io/) — computación numérica de alto rendimiento para investigación en ML

### Visualización

Representación gráfica de datos para facilitar su comprensión:

- [R](http://www.r-project.org/)
- [D3.js](http://d3js.org/)
- [Looker Studio](https://lookerstudio.google.com/) — antes Google Data Studio
- [Observable](https://observablehq.com/) — cuadernos interactivos para visualización con D3

### Panorama de tecnologías Big Data

El ecosistema de Big Data es amplio y en continua evolución. La siguiente imagen muestra el panorama de 2021 con las principales categorías y soluciones disponibles:

![2021 MAD Big Data Landscape](https://dsevilla.github.io/tcdm-public/figs/mad-big-data-landscape-2021.jpg)

## Procesamiento del Big Data: por qué Hadoop existe

Cuando los datos dejan de caber en una máquina, la solución obvia es usar varias máquinas. Pero hay un dilema económico:

- **Opción A: máquinas caras y confiables.** Si compras servidores enterprise de alta calidad, son relativamente fiables, pero muy caros. Procesar petabytes de datos exigiría una inversión gigantesca en hardware.

- **Opción B: máquinas baratas que fallan.** Comprar cientos o miles de máquinas commodity (baratas, convencionales) es mucho más económico. El problema es que **fallan continuamente**. Con miles de máquinas, algo está fallando casi todo el tiempo.

**La respuesta de Hadoop**: usar la opción B, pero asumir y gestionar automáticamente los fallos en el software. No es que Hadoop tolere los fallos *a pesar de* usar máquinas baratas; es que **la tolerancia a fallos automática es lo que hace económicamente viable** usar máquinas baratas en primer lugar.

Para que esto funcione, necesitamos:

- **Escalabilidad** con cientos o miles de máquinas y decenas de miles de discos
- **Equipos y redes de bajo coste** (aunque poco fiables y con elevadas latencias)
- **Tolerancia a fallos automática** — no como un lujo, sino como requisito fundamental
- **Facilidad de programación** — el usuario no debe pensar en fallos; el sistema los gestiona

## Problemas en la ejecución y cómo los resuelve Hadoop

La realidad de usar máquinas baratas plantea tres desafíos que Hadoop debe resolver:

### Desafío 1: Los nodos baratos fallan constantemente

**El problema:**

- Tiempo medio entre fallos (MTBF) para 1 nodo: ~3 años
- Tiempo medio entre fallos para 1.000 nodos: ~1 día

Con 10.000 nodos, el modelo proporcional sugiere que algo falla aproximadamente cada 2,6 horas. Sin tolerancia a fallos automática, el trabajo nunca terminaría.

**Cómo lo resuelve Hadoop:**

- HDFS replica cada bloque en varios nodos (típicamente 3). Cuando un nodo falla, los datos siguen disponibles en otros.
- MapReduce y YARN relanzan automáticamente tareas fallidas en otros nodos.
- El usuario no necesita intervenir: el sistema detecta el fallo, reasigna el trabajo y continúa.

### Desafío 2: Las redes baratas tienen elevada latencia

**El problema:**

- Una red de centro de datos enterprise es cara y rápida.
- Una red que conecta cientos de máquinas baratas tiene mucha latencia.
- Mover petabytes por la red es lentísimo.

**Cómo lo resuelve Hadoop:**

- **Llevar la computación a los datos**, no los datos a la computación.
- MapReduce lanza tareas map en los nodos donde están los bloques de entrada.
- El resultado local se procesa antes de transferirse por la red.
- Resultado: minimiza el movimiento de datos entre máquinas.

### Desafío 3: Programar un sistema distribuido es difícil

**El problema:**

- Sin abstracciones, los usuarios tendrían que escribir código para:
  - Particionar datos entre nodos
  - Serializar y deserializar
  - Detectar y recuperar fallos
  - Sincronizar trabajo distribuido
- Esto es tan complejo que la mayoría de errores sería no por lógica de negocio, sino por gestión distribuida.

**Cómo lo resuelve Hadoop:**

- MapReduce oculta la complejidad distribuida tras un modelo de programación simple.
- El usuario escribe dos funciones: `map()` y `reduce()`.
  - `map()` procesa pares clave-valor y emite pares intermedios.
  - `reduce()` agrupa valores por clave.
- El sistema Hadoop:
  - Distribuye el trabajo automáticamente
  - Gestiona serialización y transferencia de datos
  - Detecta y recupera fallos sin intervención del usuario
  - Coordina tareas map y reduce

**Conclusión:** La combinación de HDFS (almacenamiento tolerante a fallos), MapReduce (modelo de programación simple) y YARN (coordinación de recursos) hace posible que un usuario escriba `map()` y `reduce()` sin pensar en cientos de máquinas que fallan. Eso es lo que hace a Hadoop revolucionario.

In [ ]:
%%diapositiva resumen
# Recapitulación: las piezas antes de tocarlas
- HDFS guarda bloques replicados; YARN decide dónde se ejecutan las tareas
- MapReduce reparte un cálculo en funciones map y reduce sobre esos bloques
- Estas piezas nacieron para repartir datos y trabajo entre máquinas, no para comprar una más grande
> A partir de aquí se arranca un clúster real y se comprueba sobre él.

## Objetivos de la sesión

Al finalizar esta sesión debes saber:

- conocer la arquitectura de Hadoop y sus componentes principales;
- entender la configuración de un clúster Hadoop en contenedores Docker;
- iniciar, detener y regenerar el clúster con Docker Compose;
- distinguir el sistema de ficheros local de cada contenedor del espacio de
  nombres distribuido de HDFS;
- relacionar NameNode y DataNode con el almacenamiento de datos;
- relacionar ResourceManager y NodeManager con la ejecución de trabajos;
- consultar el estado de los tres nodos de trabajo;
- crear, copiar, mover, leer y borrar ficheros en HDFS como `luser`;
- interpretar el factor de replicación y la ubicación de los bloques;
- enviar un trabajo MapReduce a YARN y consultar su resultado;
- recuperar un clúster limpio después de una prueba fallida o de un cambio de
  permisos incorrecto.

El entorno debe quedar operativo al terminar la sesión porque se reutilizará
en sesiones posteriores con Parquet/Arrow, Spark, Trino y tablas Iceberg.

In [ ]:
%%diapositiva avance
# A continuación: preparamos el entorno y conocemos el clúster
- Comprobamos que Docker y Docker Compose funcionan en el equipo
- El laboratorio simula 4 máquinas: `namenode`, `datanode1`, `datanode2`, `datanode3`
- Vemos qué recursos anuncia cada contenedor y qué imágenes descarga Compose

## Antes de empezar

Se necesita Docker Engine o Docker Desktop con Docker Compose v2, iniciado y
con capacidad para descargar las imágenes publicadas del curso. Se recomienda
asignar al menos 8 GiB de memoria a Docker y detener otros contenedores que
no sean necesarios durante la sesión.

Las celdas de este notebook suponen que Jupyter usa como directorio de
trabajo el propio directorio del notebook (`s1/`), que es hermano de
`entorno/` en la distribución de sesiones. Por eso las rutas del entorno
se escriben como `../entorno/...`.

### Comprobar el cliente Docker

La salida debe identificar una versión de Docker. Esto sólo confirma que
existe el cliente; todavía no demuestra que el motor de contenedores esté en
ejecución.

In [ ]:
!docker --version

### Comprobar Docker Compose

El clúster está descrito como un conjunto de servicios coordinados que se
gestionan con el complemento Compose v2, invocado como parte de la orden
`docker`. Se espera una salida similar a `Docker Compose version v...`.

In [ ]:
!docker compose version

### Comprobar el motor de contenedores

Esta orden contacta con el daemon de Docker y muestra información del equipo
y de los recursos que administra. Si aparece un error como *Cannot connect to
the Docker daemon*, hay que iniciar Docker Desktop o el servicio Docker antes
de continuar: repetir las celdas de Hadoop no soluciona un daemon que no está
disponible.

In [ ]:
!docker info

## El clúster que se va a utilizar

El laboratorio simula un clúster de cuatro ordenadores mediante cuatro
contenedores que comparten una red Docker, tienen nombres DNS estables dentro
de ella y pueden comunicarse como si fueran hosts separados.

| Contenedor | Servicios | Responsabilidad principal |
| --- | --- | --- |
| `namenode` | NameNode, ResourceManager y Timeline Server | Metadatos de HDFS, coordinación de recursos de cómputo e histórico de aplicaciones YARN |
| `datanode1` | DataNode y NodeManager | Almacena bloques y ejecuta tareas |
| `datanode2` | DataNode y NodeManager | Almacena bloques y ejecuta tareas |
| `datanode3` | DataNode y NodeManager | Almacena bloques y ejecuta tareas |

El NameNode no contiene una copia completa de los ficheros de usuario:
conserva el espacio de nombres, los permisos y la relación entre ficheros,
bloques y DataNodes; los DataNodes son quienes almacenan el contenido de los
bloques. De forma análoga, el ResourceManager decide dónde se pueden ejecutar
aplicaciones, mientras que los NodeManagers ofrecen recursos y ponen en
marcha sus tareas.

El Timeline Server recibe y conserva información sobre las aplicaciones YARN
para poder consultarlas después de que terminen. En este laboratorio comparte
el contenedor del NameNode y del ResourceManager, aunque se publica con el
nombre DNS `timelineserver` para que los clientes usen una dirección
coherente con una instalación en la que fuera un host separado.

### Recursos asignados

| Contenedor | Límite Docker | Recursos anunciados a YARN |
| --- | ---: | --- |
| `namenode` | 2 vCPU y 4096 MiB | No ejecuta NodeManager |
| `datanode1` | 2 vCPU y 3072 MiB | 2 vcores y 2560 MiB |
| `datanode2` | 2 vCPU y 3072 MiB | 2 vcores y 2560 MiB |
| `datanode3` | 2 vCPU y 3072 MiB | 2 vcores y 2560 MiB |

El límite de memoria de Docker y la memoria anunciada a YARN **no son la
misma cosa**. Cada DataNode puede consumir como máximo 3072 MiB, pero el
NodeManager sólo ofrece 2560 MiB a los contenedores de las aplicaciones; los
512 MiB restantes permiten ejecutar el propio DataNode, el NodeManager y
otros procesos auxiliares. `namenode` dispone de 4096 MiB porque en
sesiones posteriores alojará también Jupyter y el driver de Spark. Los dos
vcores de cada DataNode permiten que YARN ubique más adelante un
ApplicationMaster y un executor de Spark sin dejar el nodo sin capacidad para
otras tareas.

### Imágenes del curso

Docker Compose descarga dos imágenes ya preparadas:

| Imagen | Contenedores que la utilizan |
| --- | --- |
| `dsevilla/namenode-image:26-27` | `namenode` |
| `dsevilla/datanode-image:26-27` | Los tres `datanodeN` |

Un mismo contenedor DataNode ejecuta también un NodeManager porque en Hadoop
resulta habitual acercar el procesamiento a los datos. Ambas imágenes
incorporan Hadoop 3.5.0 y Java 17; los demonios se ejecutan como `hdadmin` y
los ejercicios como `luser`. Un poco más abajo se explica cómo se construyen
estas imágenes, aunque no es necesario reconstruirlas para seguir la
sesión.

### Identidades y seguridad del laboratorio

HDFS utiliza autenticación simple: cada cliente se presenta con su propia
identidad. Cuando se ejecuta `su - luser`, Hadoop realiza las operaciones
como `luser`; con `su - hdadmin`, como el usuario administrativo. La
autenticación simple permite centrarse en HDFS y YARN, pero no demuestra
criptográficamente quién es el cliente: este entorno sólo es apropiado para
docencia local, sus puertos publicados por Compose están limitados a
`127.0.0.1` y el clúster no debe exponerse a una red externa o a Internet.

### Directorios HDFS iniciales

Durante la construcción de la imagen del NameNode se formatea HDFS y se
crean, entre otros, estos directorios:

```text
/user/hdadmin
/user/luser
/tmp
/tmp/hadoop-yarn/staging
/tmp/logs
/datalake
/datalake/raw
/datalake/raw/tpcds
/datalake/silver
/datalake/gold
/warehouse
```

`/user/luser` es el directorio de trabajo del usuario de las sesiones. Los
directorios temporales tienen *sticky bit*, de forma parecida a `/tmp` en un
sistema Unix.

`/datalake` y `/warehouse` pertenecen a `luser:hadoop` con permisos `770`.

Jupyter se ejecuta en `namenode`, y desde allí se lanzarán los clientes
PyArrow/Polars y los trabajos Spark como `luser`. Por eso los nombres
`datanode1`, `datanode2` y `datanode3` que devuelve WebHDFS son resolubles:
todos están conectados a la misma red Docker `hadoop-cluster`. `fsspec` usa
WebHDFS por HTTP y sigue los redireccionados hacia los DataNodes.

### Cómo se construyen las imágenes del curso

El alumnado utiliza imágenes ya publicadas; esta sección resume cómo se
incorporan los componentes y la configuración que se observan durante la
sesión — no es necesario reconstruirlas para seguir el resto del notebook.

#### Tres imágenes relacionadas

La construcción separa los elementos comunes de la configuración de cada
tipo de nodo:

| Imagen | Contenido añadido | Imagen de partida |
| --- | --- | --- |
| `dsevilla/hadoop-base:26-27` | Ubuntu, Java, Hadoop, utilidades y usuarios | `ubuntu:26.04` |
| `dsevilla/namenode-image:26-27` | Configuración de NameNode y ResourceManager, HDFS inicial | `hadoop-base:26-27` |
| `dsevilla/datanode-image:26-27` | Configuración de DataNode y NodeManager | `hadoop-base:26-27` |

Esta separación evita repetir la descarga e instalación de Hadoop en las dos
imágenes de servicio, y distingue qué decisiones pertenecen a todo cliente
Hadoop y cuáles son propias del nodo que guarda metadatos o del nodo que
almacena bloques.

#### Receta de la imagen base

```dockerfile
FROM ubuntu:26.04

ARG HADOOP_VERSION=3.5.0
ENV HADOOP_VERSION=${HADOOP_VERSION} \
    JAVA_HOME=/usr/lib/jvm/java-17-openjdk \
    HADOOP_HOME=/opt/bd/hadoop \
    HADOOP_CONF_DIR=/opt/bd/hadoop/etc/hadoop
```

`ARG` permite elegir la versión durante la construcción; `ENV` deja
disponibles dentro de la imagen las rutas que usarán scripts y clientes.
`$HADOOP_HOME` no apunta directamente a un directorio con el número de
versión, sino al enlace `/opt/bd/hadoop`, de modo que los comandos y la
configuración no cambian de ruta al actualizar Hadoop.

La imagen instala Java 17 sin entorno gráfico, Python, `pip`, entornos
virtuales, Maven y varias herramientas de terminal, además de `tini` (proceso
inicial del contenedor, reenvía señales y recoge procesos terminados). El
paquete `python3` es el intérprete de Ubuntu 26.04, incluido para utilidades
y sesiones posteriores; el código Python **nuevo** del curso se escribe para
Python 3.13, y no debe deducirse la versión objetivo del lenguaje a partir de
que este cliente esté instalado en la imagen Hadoop. El patrón de instalación
y limpieza es:

```sh
export DEBIAN_FRONTEND=noninteractive
apt-get update
apt-get install --no-install-recommends -y \
    ca-certificates curl tini openjdk-17-jdk-headless \
    python3 python3-venv python3-pip maven \
    libarchive-tools git make iputils-ping
apt-get clean
rm -rf /var/lib/apt/lists/* /var/cache/apt/archives/* /tmp/* /var/tmp/*
```

El Dockerfile sigue añadiendo otros paquetes, hadoop y configurando los usuarios `luser` y `hdadmin`.


#### Dockerfiles de las imágenes de servicio

```dockerfile
ARG BASE_IMAGE=dsevilla/hadoop-base:26-27
FROM ${BASE_IMAGE}

# Durante la construcción se escribe la configuración específica del NameNode.
COPY namenode.sh /tmp/
RUN sh /tmp/namenode.sh && \
    rm -f /tmp/namenode.sh

EXPOSE 9870 9871 8088 8000-10000
CMD ["/inicio.sh"]
```

El Dockerfile del DataNode es equivalente, pero escribe la configuración de
los DataNodes y del NodeManager y publica sus puertos característicos
(`9864 9865 9866 9867 50000-50200 8000-10000`). En ambos casos la
configuración se escribe **durante la construcción**, no cuando el alumnado
ejecuta `docker compose up`; la diferencia entre ambas imágenes está
principalmente en `hdfs-site.xml`, en `yarn-site.xml` y en el contenido de
`/inicio.sh`.


#### Usuarios y entorno de ejecución

```sh
groupadd --system hadoop
useradd --system --create-home --gid hadoop \
    --home-dir /opt/bd --shell /bin/bash hdadmin
useradd --system --create-home --gid hadoop \
    --home-dir /home/luser --shell /bin/bash luser
```

`hdadmin` posee la instalación de `/opt/bd` y ejecuta los demonios; `luser`
posee `/home/luser`, pertenece también al grupo `hadoop` y representa al
cliente ordinario. Compartir grupo permite preparar recursos docentes
comunes sin ejecutar los ejercicios como administrador. `/etc/profile.d/hadoop.sh`
configura todos los shells de inicio (`JAVA_HOME`, `HADOOP_HOME`,
`HADOOP_CONF_DIR`, `PATH`), por lo que `luser` puede escribir `hdfs`, `yarn` o
`mapred` sin indicar su ruta absoluta.

#### Formateo e inicialización incluidos en la imagen

El NameNode necesita un espacio de nombres formateado antes del primer
arranque. Durante la construcción de `namenode-image`:

```sh
mkdir -p /var/data/hdfs/namenode
chown -R hdadmin:hadoop /var/data/hdfs/namenode
hdfs namenode -format -force -nonInteractive
```

Este formateo se ejecuta **al construir la imagen**, no cada vez que arranca
un contenedor: repetirlo sobre un NameNode con datos generaría un nuevo
identificador de clúster y haría que los DataNodes existentes dejasen de
pertenecer a él. La construcción arranca temporalmente el NameNode, espera a
que salga del modo seguro y crea la estructura inicial:

```sh
hdfs dfs -mkdir -p \
    /user/hdadmin /user/luser \
    /tmp /tmp/logs /tmp/hadoop-yarn/staging \
    /datalake/raw/tpcds /datalake/silver /datalake/gold /warehouse
hdfs dfs -chmod 1777 /tmp /tmp/logs /tmp/hadoop-yarn/staging
hdfs dfs -chown luser /user/luser
hdfs dfs -chown -R luser:hadoop /datalake
hdfs dfs -chmod 770 \
    /datalake /datalake/raw /datalake/raw/tpcds \
    /datalake/silver /datalake/gold
hdfs dfs -chown luser:hadoop /warehouse
hdfs dfs -chmod 770 /warehouse
```

Después el NameNode se detiene de nuevo; el estado formateado y esos
directorios pasan a formar parte de la imagen publicada. El metastore y
Trino **no** se incluyen en esta imagen: se arrancarán como servicios
independientes en la misma red y accederán remotamente a HDFS (sesiones 6 y
7).

#### Arranque automático de los demonios

La imagen del NameNode termina generando `/inicio.sh`:

```sh
#!/usr/bin/env sh
set -eu
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk
export HADOOP_HOME=/opt/bd/hadoop
export HADOOP_CONF_DIR="$HADOOP_HOME/etc/hadoop"

su - hdadmin -c "JAVA_HOME=${JAVA_HOME} HADOOP_HOME=${HADOOP_HOME} HADOOP_CONF_DIR=${HADOOP_CONF_DIR} ${HADOOP_HOME}/bin/hdfs --daemon start namenode"
su - hdadmin -c "JAVA_HOME=${JAVA_HOME} HADOOP_HOME=${HADOOP_HOME} HADOOP_CONF_DIR=${HADOOP_CONF_DIR} ${HADOOP_HOME}/bin/yarn --daemon start resourcemanager"
su - hdadmin -c "JAVA_HOME=${JAVA_HOME} HADOOP_HOME=${HADOOP_HOME} HADOOP_CONF_DIR=${HADOOP_CONF_DIR} ${HADOOP_HOME}/bin/yarn --daemon start timelineserver"

# Bucle infinito
exec tail -f /dev/null
```

El Dockerfile declara este script como `CMD`, así que se ejecuta al crear el
contenedor; los demonios se inician como `hdadmin`, no como `root`. El
proceso `tail` mantiene activo el contenedor tras dejar los servicios en
segundo plano, y `tini` (declarado en la imagen base) permanece como proceso
inicial gestionando correctamente las señales. En la salida de `jps`, el
Timeline Server aparece con el nombre de clase Java
`ApplicationHistoryServer`, aunque las órdenes de Hadoop lo gestionen como
servicio `timelineserver`. Los DataNodes generan una versión equivalente que
arranca `datanode` y `nodemanager` en vez de `namenode` y
`resourcemanager`/`timelineserver`. Esto explica por qué `docker compose
start` vuelve a poner en marcha los servicios sin que el alumnado ejecute
manualmente `hdfs --daemon start` en cuatro terminales.

Las imágenes publicadas incluyen variantes `linux/amd64` y `linux/arm64`;
Docker selecciona automáticamente la adecuada para el equipo al descargar
la etiqueta `26-27`.

## Preparar la red del laboratorio

El fichero Compose declara `hadoop-cluster` como una **red externa**. Esto
hace que su ciclo de vida sea independiente del clúster Hadoop y permitirá
conectar más adelante otros contenedores —Trino, PostgreSQL, Hive
Metastore— sin incluirlos en el mismo fichero Compose.

La siguiente orden primero comprueba si la red ya existe y sólo la crea si
hace falta; crear la red no arranca ningún contenedor, únicamente prepara el
espacio de red y el DNS interno que compartirán los servicios. Si no imprime
nada es que la red ya existía.

In [ ]:
!docker network inspect hadoop-cluster >/dev/null 2>&1 || docker network create hadoop-cluster

In [ ]:
%%diapositiva resumen
# Recapitulación: el entorno está listo para ser lanzado
- Docker, Compose y el motor de contenedores responden correctamente
- Conocemos los 4 contenedores, sus servicios y los recursos que anuncian a YARN
- La red `hadoop-cluster` ya existe y conectará también servicios futuros (Trino, Hive Metastore)

In [ ]:
%%diapositiva avance
# A continuación: arrancamos el clúster
- Cuatro contenedores: `namenode`, `datanode1`, `datanode2`, `datanode3`
- Cada uno representa una máquina distinta de un clúster real
- Vamos a comprobar que Docker los levanta y que Hadoop los reconoce

## Arrancar el clúster

La siguiente orden lee el fichero Compose, descarga las imágenes que falten
en el equipo (`-d` es *detached*: los contenedores quedan en segundo plano y
el terminal vuelve a estar disponible) y crea los cuatro contenedores. La
primera ejecución puede tardar varios minutos, especialmente si hay que
descargar imágenes para otra arquitectura de procesador; las siguientes
suelen ser mucho más rápidas.

In [ ]:
!docker compose -f ../entorno/compose-hadoop-cluster.yml up -d

### Ver el estado de los contenedores

Se espera que `namenode` y los tres `datanodeN` aparezcan en estado
`running`. Que un contenedor esté en ejecución sólo significa que su proceso
principal sigue vivo: los demonios Hadoop necesitan además unos segundos para
arrancar y registrarse. Las comprobaciones de la siguiente sección son las
que confirman que el clúster está realmente preparado.

In [ ]:
!docker compose -f ../entorno/compose-hadoop-cluster.yml ps

### Observar el clúster desde el navegador

Cuando el clúster esté listo, puedes abrir estas direcciones desde el
navegador del equipo anfitrión (usan `localhost` porque Docker publica esos
puertos sólo en la interfaz local del equipo; no hay que sustituir
`localhost` por la dirección de un DataNode):

- NameNode/HDFS: <http://localhost:9870> (prueba también **Utilities → Browse
  the file system** para relacionar las órdenes de terminal con los ficheros
  visibles desde la interfaz)
- ResourceManager/YARN: <http://localhost:8088>
- YARN Timeline Server: <http://localhost:8188/applicationhistory/> (la barra
  final es necesaria en esta versión de Hadoop; sin ella el servidor responde
  `404 Not Found`)

El Timeline Server no añade un quinto contenedor: el alias Docker
`timelineserver` apunta a `namenode`, donde el proceso
`ApplicationHistoryServer` se ejecuta junto al NameNode y al ResourceManager,
con su propio almacén LevelDB. Este servicio **no** es el `JobHistoryServer`
de MapReduce: el Timeline Server recoge el histórico genérico de
aplicaciones YARN (el que se usa en esta sesión), mientras que el
JobHistoryServer es otro servicio para los eventos detallados de trabajos
MapReduce.

Estas interfaces son una ayuda de observación; las comprobaciones
reproducibles se hacen también desde este notebook.

## Comprobar HDFS y YARN

`hdfs dfsadmin -report` pregunta al NameNode por los DataNodes registrados y
debe mostrar **3 live datanodes**. Si falla poco después del arranque, espera
15-30 segundos y repite únicamente esta celda: es posible que los tres nodos
todavía no se hayan registrado.

In [ ]:
!docker exec namenode su - hdadmin -c 'hdfs dfsadmin -report'

YARN administra recursos de procesamiento, no bloques HDFS, y sus nodos de
trabajo se consultan por separado. La salida esperada contiene tres
NodeManagers en estado `RUNNING`, cada uno anunciando dos vcores y la
memoria configurada. Es posible que HDFS tenga tres DataNodes vivos y YARN no
tenga tres NodeManagers, o al contrario, porque son servicios diferentes
aunque compartan contenedores:

- `hdfs dfsadmin -report` responde a «¿dónde se pueden guardar bloques?»;
- `yarn node -list -all` responde a «¿dónde se pueden ejecutar tareas?».

In [ ]:
!docker exec namenode su - hdadmin -c 'yarn node -list -all'

### Relacionar servicios y procesos con `jps`

`jps` muestra los procesos Java visibles para el usuario que lo ejecuta. Los
demonios pertenecen a `hdadmin`, por lo que hay que usar esa identidad para
obtener una comprobación fiable. En el NameNode se espera encontrar
`NameNode`, `ResourceManager` y `ApplicationHistoryServer` (además del propio
proceso `Jps`, que sólo existe mientras se hace la consulta).

In [ ]:
!docker exec namenode su - hdadmin -c 'jps'

En cada DataNode se espera encontrar `DataNode` y `NodeManager`.

In [ ]:
!docker exec datanode1 su - hdadmin -c 'jps'

In [ ]:
!docker exec datanode2 su - hdadmin -c 'jps'

In [ ]:
!docker exec datanode3 su - hdadmin -c 'jps'

| Proceso | Qué mantiene o ejecuta |
| --- | --- |
| `NameNode` | Espacio de nombres, permisos y localización de los bloques HDFS |
| `DataNode` | Bloques de datos guardados en el almacenamiento local del nodo |
| `ResourceManager` | Vista global de recursos y planificación de aplicaciones |
| `ApplicationHistoryServer` | Histórico y consulta de información de aplicaciones YARN (Timeline Server) |
| `NodeManager` | Contenedores YARN y recursos disponibles en un nodo |

Los identificadores numéricos de proceso no tienen por qué coincidir entre
contenedores ni mantenerse tras un reinicio; lo relevante son los nombres de
los demonios.

### Consultar el Timeline Server desde la red interna

El ResourceManager sabe si una aplicación está ejecutándose, ha terminado o
ha fallado. El Timeline Server añade una perspectiva **histórica**: recibe
los eventos y métricas que publican el ResourceManager y las aplicaciones, y
los conserva en su almacén local. La siguiente celda comprueba que la API
HTTP responde (la salida es un documento JSON) usando el alias interno, lo
que también verifica que el nombre `timelineserver` resuelve al contenedor
correcto. Más adelante, tras ejecutar WordCount, se volverá a esta interfaz
para localizar la aplicación ya finalizada.

In [ ]:
!docker exec namenode curl --fail --silent http://timelineserver:8188/ws/v1/timeline/

In [ ]:
%%diapositiva resumen
# Recapitulación: el clúster está vivo
- 3 DataNodes replicando bloques, 3 NodeManagers ofreciendo recursos
- El NameNode manda en el espacio de nombres; no guarda los datos
- El ResourceManager decide dónde se ejecutan las tareas, no dónde viven los bloques
> A partir de aquí se usa ese clúster: primero HDFS, después MapReduce.

In [ ]:
%%diapositiva avance
# A continuación: exploramos el clúster como `luser`
- Qué herramientas trae la imagen y desde dónde se resuelven
- El espacio de nombres HDFS: `/user`, `/datalake`, `/warehouse`
- Replicación, tamaño de bloque y su relación con los splits de MapReduce

## Explorar las herramientas instaladas

Las imágenes no son cajas opacas. Aunque no haga falta construirlas, conviene
comprobar qué ejecutables contienen y desde dónde se invocan. Todas las
órdenes siguientes se ejecutan dentro de `namenode` como `luser`: será la
identidad habitual para trabajar con datos y, más adelante, para iniciar
Jupyter y Spark.

In [ ]:
!docker exec namenode su - luser -c 'id'

`command -v` muestra el ejecutable que resolvería el shell a través de
`PATH`, sin llegar a ejecutar el programa. La imagen incluye Java 17, y
Python/Maven para sesiones posteriores.

In [ ]:
!docker exec namenode su - luser -c 'command -v java && java -version'

In [ ]:
!docker exec namenode su - luser -c 'command -v python3 && python3 --version'

In [ ]:
!docker exec namenode su - luser -c 'command -v pip3'

In [ ]:
!docker exec namenode su - luser -c 'command -v mvn'

Los clientes de Hadoop deben resolverse todos dentro de `/opt/bd/hadoop/bin`,
sin que haga falta escribir la ruta completa: la instalación se expone
mediante variables de entorno (`$HADOOP_HOME`, `$PATH`) definidas en
`/etc/profile.d/hadoop.sh`.

In [ ]:
!docker exec namenode su - luser -c 'command -v hadoop; command -v hdfs; command -v yarn; command -v mapred'

La primera línea de `hadoop version` debe indicar Hadoop 3.5.0; el resto de
la salida aporta datos de compilación y de la JVM, útiles cuando un mensaje
de error o una documentación externa dependen de una versión concreta.

In [ ]:
!docker exec namenode su - luser -c 'hadoop version'

In [ ]:
!docker exec namenode su - luser -c 'printf "%s\n" "$HADOOP_HOME"'

## Explorar HDFS como `luser`

A partir de aquí todas las operaciones ordinarias de datos se ejecutan como
`luser`; `hdadmin` queda reservado para consultas administrativas (`fsck`,
`dfsadmin`). Trabajar con la identidad correcta permite detectar problemas de
permisos que quedarían ocultos si todo se hiciera como administrador.

La raíz de HDFS **no** es la raíz de Linux del contenedor: es el espacio de
nombres distribuido. Deberían aparecer `/user`, `/tmp`, `/datalake` y
`/warehouse`, cada uno con su propietario, grupo, permisos, tamaño y fecha.

In [ ]:
!docker exec namenode su - luser -c 'hdfs dfs -ls /'

Cada usuario puede tener su área bajo `/user` (una convención parecida a
`/home/<usuario>` en Unix, aunque ambos sistemas de ficheros sean
independientes). Debe aparecer al menos `/user/hdadmin` y `/user/luser`.

In [ ]:
!docker exec namenode su - luser -c 'hdfs dfs -ls /user'

`/datalake/raw/tpcds` es donde una sesión posterior materializará TPC-DS en
Parquet. `/datalake/silver` y `/datalake/gold` reservan las siguientes capas
medallion, y `/warehouse` queda para tablas gobernadas por catálogo. Estas
rutas deben pertenecer a `luser:hadoop`. En esta sesión sólo se comprueba
que existen y son accesibles.

In [ ]:
!docker exec namenode su - luser -c 'hdfs dfs -ls -d /datalake /datalake/raw /datalake/raw/tpcds /datalake/silver /datalake/gold'

In [ ]:
!docker exec namenode su - luser -c 'hdfs dfs -ls -d /warehouse'

## Replicación y tamaño de bloque

El factor de replicación por defecto se consulta con `getconf`, sin
necesidad de editar ficheros de configuración. El valor esperado es `3`: HDFS
intentará mantener tres copias de cada bloque, una en cada DataNode
disponible. Replicar tres veces no crea tres nombres de fichero ni hace que
`hdfs dfs -ls` muestre tres entradas — las réplicas son una decisión interna
de almacenamiento.

In [ ]:
!docker exec namenode su - luser -c 'hdfs getconf -confKey dfs.replication'

El tamaño de bloque configurado corresponde a 64 MiB. Un bloque HDFS no
equivale al bloque físico de un disco: es una unidad lógica grande, elegida
para reducir metadatos y favorecer lecturas secuenciales de datos masivos.
Los ficheros de esta sesión son mucho más pequeños y ocuparán un único
bloque cada uno; en conjuntos de datos grandes, HDFS divide cada fichero en
varios bloques que pueden almacenarse y procesarse en nodos diferentes.

In [ ]:
!docker exec namenode su - luser -c 'hdfs getconf -confKey dfs.blocksize'

In [ ]:
%%diapositiva avance
# A continuación: la configuración real de Hadoop
- Las propiedades de cada fichero, tal y como se escribieron al construir la imagen
- `core-site.xml`, `hdfs-site.xml`, `yarn-site.xml` y `mapred-site.xml`, fichero a fichero
- Qué cambia entre `namenode` y un `datanodeN` en cada fichero

## Comprobar la configuración efectiva de Hadoop

Los cuatro ficheros principales de configuración de Hadoop
(`core-site.xml`, `hdfs-site.xml`, `yarn-site.xml` y `mapred-site.xml`) viven
en `$HADOOP_CONF_DIR`, es decir `/opt/bd/hadoop/etc/hadoop`, y se escriben al
construir la imagen. Aquí se muestran las propiedades de cada uno, tal y como
quedaron escritas (el script de la imagen añade además comentarios que aquí
se omiten), y se explica justo debajo el significado de las más
importantes.

`core-site.xml` es común a NameNode y DataNodes, y fija `fs.defaultFS`: hace
que una ruta como `/user/luser` se interprete como
`hdfs://namenode:9000/user/luser`.

```xml
<?xml version="1.0"?>
<configuration>
  <property>
    <name>fs.defaultFS</name>
    <value>hdfs://namenode:9000/</value>
    <final>true</final>
  </property>
  <property>
    <name>hadoop.tmp.dir</name>
    <value>/var/tmp/hadoop-${user.name}</value>
    <final>true</final>
  </property>
</configuration>
```

El puerto `9000` de `fs.defaultFS` es el endpoint RPC que usan los clientes
HDFS (`hdfs dfs`, `fsspec`, Spark); no debe confundirse con la interfaz web
`9870`. `hadoop.tmp.dir` fija dónde escribe cada identidad sus temporales
locales, separados gracias a `${user.name}`, que Hadoop resuelve en tiempo
de ejecución. Ambas propiedades llevan `<final>true</final>`: ningún trabajo
puede sustituirlas con su propia configuración, porque son decisiones
estructurales de este laboratorio, no ajustes que deba tocar quien lo usa.

`hdfs-site.xml` cambia según el tipo de nodo. La diferencia principal entre
el NameNode y un DataNode será `dfs.namenode.name.dir` frente a
`dfs.datanode.data.dir`: uno señala dónde se guardan **metadatos** y el otro
dónde se guardan **bloques**.

```xml
<?xml version="1.0"?>
<configuration>
  <property>
    <name>dfs.replication</name>
    <value>3</value>
    <final>true</final>
  </property>
  <property>
    <name>dfs.blocksize</name>
    <value>64m</value>
    <final>true</final>
  </property>
  <property>
    <name>dfs.namenode.name.dir</name>
    <value>file:///var/data/hdfs/namenode</value>
    <final>true</final>
  </property>
  <property>
    <name>dfs.image.compress</name>
    <value>true</value>
  </property>
  <property>
    <name>dfs.webhdfs.enabled</name>
    <value>true</value>
  </property>
  <property>
    <name>hadoop.http.authentication.type</name>
    <value>simple</value>
  </property>
  <property>
    <name>hadoop.http.filter.initializers</name>
    <value></value>
  </property>
  <property>
    <name>dfs.namenode.http-address</name>
    <value>namenode:9870</value>
  </property>
</configuration>
```

```xml
<?xml version="1.0"?>
<configuration>
  <property>
    <name>dfs.replication</name>
    <value>3</value>
    <final>true</final>
  </property>
  <property>
    <name>dfs.blocksize</name>
    <value>64m</value>
    <final>true</final>
  </property>
  <property>
    <name>dfs.datanode.data.dir</name>
    <value>file:///var/data/hdfs/datanode</value>
    <final>true</final>
  </property>
  <property>
    <name>dfs.webhdfs.enabled</name>
    <value>true</value>
  </property>
</configuration>
```

`dfs.namenode.name.dir` y `dfs.datanode.data.dir` son las dos propiedades
que distinguen ambos ficheros: la primera es una ruta **local** del
contenedor `namenode` donde vive la imagen del espacio de nombres y el
registro de ediciones; la segunda es una ruta local de cada `datanodeN`
donde se guardan sus bloques. Aunque el texto de la ruta coincida entre los
tres DataNodes, cada uno la resuelve en su propio sistema de ficheros: no la
comparten. Por eso no se usa RAID para la redundancia de los bloques de
HDFS —esa responsabilidad ya la cumple la replicación entre nodos—,
mientras que perder el disco del NameNode sí sería grave, porque ahí sólo
hay una copia de los metadatos.

`dfs.webhdfs.enabled=true` activa la API HTTP de HDFS, que usarán `fsspec`,
PyArrow y Polars en una sesión posterior para leer datos sin pasar por el
cliente `hdfs`. `dfs.namenode.http-address` es la dirección que publica esa
interfaz de administración, la misma que se abre en <http://localhost:9870>.

`compose-hadoop-cluster.yml` monta cada una de estas dos rutas sobre un
volumen Docker con nombre (`namenode-data`, `datanode1-data`,
`datanode2-data`, `datanode3-data`), no sobre la capa de escritura del
contenedor. Por eso el espacio de nombres y los bloques sobreviven a
`docker compose down` y a la recreación de los contenedores: solo
desaparecen si se borra explícitamente el volumen (ver «Detener, limpiar y
recuperar el entorno», más adelante en esta sesión).

`yarn-site.xml` configura el ResourceManager (en `namenode`) y los
NodeManagers (en cada `datanodeN`). Compara los límites de planificador del
ResourceManager con los recursos que anuncia un NodeManager concreto.

Del `yarn-site.xml` de los DataNodes se muestra, tras el del NameNode, sólo el
bloque de propiedades propias del NodeManager: las otras ocho (host del
ResourceManager, Timeline Server y agregación de logs) son idénticas a las del
NameNode.

```xml
<?xml version="1.0"?>
<configuration>
  <property>
    <name>yarn.resourcemanager.hostname</name>
    <value>resourcemanager</value>
    <final>true</final>
  </property>
  <property>
    <name>yarn.timeline-service.enabled</name>
    <value>true</value>
    <final>true</final>
  </property>
  <property>
    <name>yarn.timeline-service.hostname</name>
    <value>timelineserver</value>
    <final>true</final>
  </property>
  <property>
    <name>yarn.timeline-service.address</name>
    <value>timelineserver:10200</value>
    <final>true</final>
  </property>
  <property>
    <name>yarn.timeline-service.webapp.address</name>
    <value>timelineserver:8188</value>
    <final>true</final>
  </property>
  <property>
    <name>yarn.timeline-service.bind-host</name>
    <value>0.0.0.0</value>
    <final>true</final>
  </property>
  <property>
    <name>yarn.system-metrics-publisher.enabled</name>
    <value>true</value>
    <final>true</final>
  </property>
  <property>
    <name>yarn.timeline-service.generic-application-history.enabled</name>
    <value>true</value>
    <final>true</final>
  </property>
  <property>
    <name>yarn.timeline-service.store-class</name>
    <value>org.apache.hadoop.yarn.server.timeline.LeveldbTimelineStore</value>
    <final>true</final>
  </property>
  <property>
    <name>yarn.timeline-service.leveldb-timeline-store.path</name>
    <value>/var/data/yarn/timeline</value>
    <final>true</final>
  </property>
  <property>
    <name>yarn.timeline-service.http-authentication.type</name>
    <value>simple</value>
    <final>true</final>
  </property>
  <property>
    <name>yarn.timeline-service.http-authentication.simple.anonymous.allowed</name>
    <value>true</value>
    <final>true</final>
  </property>
  <property>
    <name>yarn.log-aggregation-enable</name>
    <value>true</value>
  </property>
  <property>
    <name>yarn.nodemanager.remote-app-log-dir</name>
    <value>/tmp/logs</value>
  </property>
  <property>
    <name>yarn.nodemanager.log-aggregation.compression-type</name>
    <value>gz</value>
  </property>
  <property>
    <name>yarn.scheduler.maximum-allocation-vcores</name>
    <value>2</value>
  </property>
  <property>
    <name>yarn.scheduler.minimum-allocation-mb</name>
    <value>256</value>
  </property>
  <property>
    <name>yarn.scheduler.maximum-allocation-mb</name>
    <value>2560</value>
  </property>
</configuration>
```

```xml
<property>
  <name>yarn.nodemanager.resource.detect-hardware-capabilities</name>
  <value>false</value>
</property>
<property>
  <name>yarn.nodemanager.resource.cpu-vcores</name>
  <value>2</value>
</property>
<property>
  <name>yarn.nodemanager.resource.memory-mb</name>
  <value>2560</value>
</property>
<property>
  <name>yarn.nodemanager.vmem-check-enabled</name>
  <value>false</value>
</property>
<property>
  <name>yarn.nodemanager.aux-services</name>
  <value>mapreduce_shuffle</value>
</property>
<property>
  <name>yarn.nodemanager.aux-services.mapreduce_shuffle.class</name>
  <value>org.apache.hadoop.mapred.ShuffleHandler</value>
</property>
```

`resourcemanager` y `timelineserver` son alias de red del propio contenedor
`namenode`, para separar el nombre de cada servicio de su papel en HDFS
aunque compartan contenedor; el puerto RPC `10200` del Timeline Server sólo
se usa dentro de la red Docker, y su interfaz HTTP se publica en el host
como `http://localhost:8188`. La propiedad que permite que el
ResourceManager publique información de aplicaciones es
`yarn.system-metrics-publisher.enabled` (en Hadoop 3.5.0 la variante con el
prefijo `resourcemanager.` está deprecada). Esta es la clave que se usa en
el laboratorio; no se debe sustituir por una advertencia de otra versión.
La agregación de logs
(`yarn.log-aggregation-enable`) copia a `/tmp/logs` de HDFS los logs de los
contenedores, comprimidos como TFile con `gz`: no se puede leer ese
resultado con un `cat` directo, hace falta
`yarn logs -applicationId <application_id>`, que conoce el formato. Los
límites del planificador (`yarn.scheduler.maximum-allocation-*`) impiden
pedir un contenedor mayor que los recursos que anuncia un DataNode.

El Timeline Server guarda su histórico con `LeveldbTimelineStore`, en la ruta
local `/var/data/yarn/timeline` del contenedor `namenode`
(`yarn.timeline-service.leveldb-timeline-store.path`). A diferencia de
`/var/data/hdfs/namenode`, esa ruta no está montada en un volumen con nombre:
el histórico de aplicaciones desaparece con `docker compose down` aunque los
datos HDFS se conserven. `yarn.timeline-service.bind-host=0.0.0.0` hace que el
servicio escuche en todas las interfaces del contenedor —Compose publica
después ese puerto sólo en `127.0.0.1`—, y las dos propiedades
`http-authentication` (tipo `simple`, con acceso anónimo permitido) explican
que la interfaz <http://localhost:8188/applicationhistory/> se abra en el
navegador sin credenciales, en línea con la autenticación simple del resto del
laboratorio.

En el `yarn-site.xml` de los NodeManagers,
`yarn.nodemanager.resource.detect-hardware-capabilities=false` evita que
YARN interprete los recursos del host en vez de los límites que se quieren
enseñar: cada NodeManager anuncia explícitamente 2 vcores y 2560 MiB,
dejando 512 MiB del límite Docker (3072 MiB) para el propio DataNode, el
NodeManager y otros procesos. La comprobación de memoria virtual se
desactiva (`yarn.nodemanager.vmem-check-enabled=false`) para evitar falsos
positivos por cómo Linux y las JVM reservan espacio de direcciones dentro
de un contenedor; esto no elimina el límite real de memoria de Docker ni el
presupuesto que YARN asigna a sus contenedores. `mapreduce_shuffle` es el
servicio auxiliar que permite a los reducers obtener de cada nodo las
salidas intermedias de los mapas: sin él, YARN podría iniciar procesos,
pero un trabajo MapReduce no completaría su fase de *shuffle*.

`mapred-site.xml` es común a las dos imágenes de servicio: los NodeManagers
son quienes ejecutan allí el ApplicationMaster, las tareas `map` y las
tareas `reduce`.

```xml
<?xml version="1.0"?>
<configuration>
  <property>
    <name>mapreduce.framework.name</name>
    <value>yarn</value>
    <final>true</final>
  </property>
  <property>
    <name>yarn.app.mapreduce.am.env</name>
    <value>HADOOP_MAPRED_HOME=${HADOOP_HOME}</value>
  </property>
  <property>
    <name>yarn.app.mapreduce.am.resource.cpu-vcores</name>
    <value>1</value>
  </property>
  <property>
    <name>yarn.app.mapreduce.am.resource.mb</name>
    <value>512</value>
  </property>
  <property>
    <name>mapreduce.job.heap.memory-mb.ratio</name>
    <value>0.8</value>
  </property>
  <property>
    <name>mapreduce.map.env</name>
    <value>HADOOP_MAPRED_HOME=${HADOOP_HOME}</value>
  </property>
  <property>
    <name>mapreduce.map.cpu.vcores</name>
    <value>1</value>
  </property>
  <property>
    <name>mapreduce.map.java.opts</name>
    <value>-Xmx512M</value>
  </property>
  <property>
    <name>mapreduce.map.memory.mb</name>
    <value>768</value>
  </property>
  <property>
    <name>mapreduce.reduce.env</name>
    <value>HADOOP_MAPRED_HOME=${HADOOP_HOME}</value>
  </property>
  <property>
    <name>mapreduce.reduce.cpu.vcores</name>
    <value>1</value>
  </property>
  <property>
    <name>mapreduce.reduce.java.opts</name>
    <value>-Xmx512M</value>
  </property>
  <property>
    <name>mapreduce.reduce.memory.mb</name>
    <value>768</value>
  </property>
</configuration>
```

`mapreduce.framework.name=yarn` evita la ejecución local del trabajo: los
recursos siempre se piden al clúster. El ApplicationMaster reserva 1 vcore
y 512 MiB; cada tarea `map` y `reduce` solicita 1 vcore y un contenedor de
768 MiB, pero limita el *heap* de su JVM a 512 MiB (`-Xmx512M`). El ratio
`mapreduce.job.heap.memory-mb.ratio=0.8` sería una regla de inferencia si
faltaran estos valores explícitos; aquí no se usa para obtener los 614,4
MiB hipotéticos. La diferencia deja memoria para la propia JVM, bibliotecas nativas y otros
costes fuera del *heap*. `HADOOP_MAPRED_HOME` se propaga a los procesos
remotos (`*.env`) para que cualquier DataNode localice las bibliotecas de
MapReduce.

También se puede pedir a Hadoop una propiedad ya interpretada —incluyendo
sustituciones y valores predeterminados— en lugar de leer el XML a mano.
Leer el XML explica **qué se escribió al construir la imagen**; consultar la
propiedad mediante Hadoop confirma **qué valor está utilizando el proceso**.
Ambas formas de inspección son complementarias.

In [ ]:
!docker exec namenode su - luser -c 'hdfs getconf -confKey fs.defaultFS'

In [ ]:
%%diapositiva resumen
# Recapitulación: la configuración explica lo ya observado
- `fs.defaultFS=hdfs://namenode:9000/` explica por qué no hace falta indicar una URL al especificar un fichero en HDFS
- `dfs.replication=3` y `dfs.blocksize=64m` son los valores que ya comprobamos con `getconf`
- `mapreduce.framework.name=yarn` obliga a MapReduce a pedir recursos al clúster, no a ejecutarse en local

In [ ]:
%%diapositiva avance
# A continuación: ficheros de verdad en HDFS
- Creamos el área de trabajo `/user/luser/s1` y subimos ficheros locales
- Copiamos, movemos, leemos y descargamos con `hdfs dfs`
- Comprobamos bloques y réplicas con `du` y `fsck`

## Crear el área de trabajo de la sesión

Los datos de esta sesión se aíslan bajo `/user/luser/s1/input`. `-p` crea
también los directorios intermedios y no falla si ya existen, por lo que se
puede repetir esta celda al volver a hacer la sesión.

In [ ]:
!docker exec namenode su - luser -c 'hdfs dfs -mkdir -p /user/luser/s1/input'

In [ ]:
!docker exec namenode su - luser -c 'hdfs dfs -ls -R /user/luser/s1'

## Operaciones básicas con ficheros en HDFS

Primero se crean tres ficheros **locales** en `/tmp` del contenedor
`namenode`: todavía no están replicados, no son visibles desde los
DataNodes mediante HDFS y desaparecerán si se elimina el contenedor. Esta
distinción entre ruta local y ruta HDFS será importante durante todo el
curso.

Los tres ficheros comparten varias palabras a propósito, para que el
resultado de WordCount permita comprobar tanto recuentos repetidos como
palabras que sólo aparecen una vez.

In [ ]:
!docker exec namenode su - luser -c 'printf "hadoop hdfs mapreduce\nhadoop yarn\n" > /tmp/s1-a.txt'

In [ ]:
!docker exec namenode su - luser -c 'printf "hdfs hadoop\nyarn recursos\n" > /tmp/s1-b.txt'

In [ ]:
!docker exec namenode su - luser -c 'printf "mapreduce datos hadoop\ndatos datos\n" > /tmp/s1-c.txt'

Se puede comprobar el contenido local del primer fichero con `cat`, sin usar todavía el cliente HDFS:

In [ ]:
!docker exec namenode su - luser -c 'cat /tmp/s1-a.txt'

### Subir los ficheros a HDFS

`-put -f` transfiere el fichero local al directorio HDFS indicado; `-f`
permite reemplazar un fichero HDFS del mismo nombre si se repite la
sesión. Tener varios ficheros de entrada permite a Hadoop crear varias
tareas `map`: con datos tan pequeños el rendimiento no mejora, pero la
estructura ayuda a observar que una entrada puede dividirse en trabajos
independientes.

In [ ]:
!docker exec namenode su - luser -c 'hdfs dfs -put -f /tmp/s1-a.txt /user/luser/s1/input/'

In [ ]:
!docker exec namenode su - luser -c 'hdfs dfs -put -f /tmp/s1-b.txt /user/luser/s1/input/'

In [ ]:
!docker exec namenode su - luser -c 'hdfs dfs -put -f /tmp/s1-c.txt /user/luser/s1/input/'

### Listar y leer los datos distribuidos

Se esperan tres entradas; `-h` presenta los tamaños en formato legible. El
factor de replicación no multiplica el tamaño lógico mostrado en esta lista.

In [ ]:
!docker exec namenode su - luser -c 'hdfs dfs -ls -h /user/luser/s1/input'

`-cat` envía los bytes recuperados desde los DataNodes a la salida estándar; no crea una copia local permanente.

In [ ]:
!docker exec namenode su - luser -c 'hdfs dfs -cat /user/luser/s1/input/s1-a.txt'

### Copiar, mover y descargar ficheros

`-cp` copia entre dos rutas que pertenecen a HDFS; no interviene el sistema
de ficheros local del contenedor.

In [ ]:
!docker exec namenode su - luser -c 'hdfs dfs -cp /user/luser/s1/input/s1-a.txt /user/luser/s1/s1-a-copia.txt'

`-mv` cambia la ruta dentro de HDFS. Dentro de un mismo espacio de nombres, renombrar suele ser una operación de metadatos del NameNode: no hace falta volver a copiar los bloques.

In [ ]:
!docker exec namenode su - luser -c 'hdfs dfs -mv /user/luser/s1/s1-a-copia.txt /user/luser/s1/s1-a-renombrado.txt'

El fichero renombrado queda directamente bajo `/user/luser/s1`, fuera de `input`, así que MapReduce no lo contará más adelante.

In [ ]:
!docker exec namenode su - luser -c 'hdfs dfs -ls /user/luser/s1'

`-get` es la operación inversa a `-put`: descarga un fichero de HDFS al sistema local del contenedor. Tras esto, `/tmp/s1-descargado.txt` es una copia local que se puede comprobar con `cat`, sin usar ya el cliente HDFS.

In [ ]:
!docker exec namenode su - luser -c 'hdfs dfs -get -f /user/luser/s1/s1-a-renombrado.txt /tmp/s1-descargado.txt'

In [ ]:
!docker exec namenode su - luser -c 'cat /tmp/s1-descargado.txt'

### Espacio, bloques y réplicas

`-du -h` muestra el tamaño lógico y el espacio ocupado por las réplicas. Con
factor de replicación tres, el segundo valor debería ser aproximadamente el
triple del primero para estos ficheros, aunque sean muy pequeños.

In [ ]:
!docker exec namenode su - luser -c 'hdfs dfs -du -h /user/luser/s1/input'

`fsck` consulta el estado de bajo nivel de bloques y réplicas como administrador. La última línea debe indicar `Status: HEALTHY`; las ubicaciones muestran en qué DataNodes se guarda cada réplica y se pueden comparar con los tres nodos obtenidos antes con `dfsadmin -report`.

In [ ]:
!docker exec namenode su - hdadmin -c 'hdfs fsck /user/luser/s1/input -files -blocks -locations'

In [ ]:
%%diapositiva resumen
# Recapitulación: HDFS ya tiene datos propios
- `/user/luser/s1/input` contiene tres ficheros replicados y verificados con `fsck`
- `-put`, `-cp`, `-mv` y `-get` distinguen siempre entre ruta local y ruta HDFS
- Con datos ya en HDFS, toca procesarlos: MapReduce

In [ ]:
%%diapositiva avance
# A continuación: el modelo MapReduce, antes de lanzarlo
- `map(K1, V1) -> list(K2, V2)`: convierte cada línea de entrada en pares intermedios
- Hadoop agrupa esos pares por clave durante el *shuffle*
- `reduce(K2, list(V2)) -> (K3, list(V3))`: combina los valores de cada clave
- WordCount aplicará este modelo con datos reales sobre el clúster

## El modelo de programación MapReduce

Antes de enviar un trabajo a YARN conviene fijar el modelo que MapReduce
oculta detrás de esa orden. El programador sólo escribe dos funciones:

- **map**: `map(K1, V1) -> list(K2, V2)`. Recibe un par clave/valor de la
  entrada y produce una lista de pares intermedios. En WordCount, `K1` no se
  usa, `V1` es una línea de texto y cada palabra de esa línea produce un par
  intermedio `(palabra, 1)`.
- **reduce**: `reduce(K2, list(V2)) -> (K3, list(V3))`. Recibe todos los
  valores intermedios asociados a una misma clave —Hadoop los agrupa durante
  la fase de *shuffle*— y produce el resultado final. En WordCount,
  `list(V3)` contiene un único valor: la suma de los `1` recibidos para esa
  palabra.

Dos piezas adicionales optimizan ese esquema sin cambiar el resultado:

- Un **combinador** (*combiner*) agrega localmente, en el mismo nodo que
  ejecutó el `map`, los valores de una misma clave antes de enviarlos por
  red. Reduce el tráfico del *shuffle*, pero sólo es correcto si la función
  es conmutativa y asociativa —sumar recuentos de palabras lo es—.
- Un **particionador** decide a qué tarea `reduce` va cada clave, típicamente
  con `hash(K) mod R`, donde `R` es el número de reducers. Garantiza que
  todas las apariciones de una misma clave llegan al mismo reducer, sin lo
  cual no se podría sumar correctamente su recuento.

MapReduce también está diseñado para que un fallo aislado no tumbe el
trabajo completo. Un proceso maestro detecta mediante *heartbeats* qué
tareas siguen vivas; si una tarea falla, se reintenta en otro nodo, y si una
tarea concreta no avanza —una tarea *rezagada* o *straggler*— puede lanzarse
una segunda copia en paralelo en otro nodo y quedarse con la que termine
antes (*ejecución especulativa*). Las tareas `map` no dependen unas de otras,
así que pueden reejecutarse sin coordinación; las tareas `reduce` pueden
recuperarse porque las salidas de `map` quedan en disco local, no sólo en
memoria.

La siguiente figura resume el flujo completo, incluida la fase de *shuffle*
que agrupa por clave entre la fase `map` y la fase `reduce`:

<p align='center'><img src='https://dsevilla.github.io/tcdm-public/figs/mapreduce-flujo.svg' alt='Flujo completo de MapReduce: las tareas map leen las entradas del sistema de ficheros distribuido, el barajado agrupa y ordena por clave y las tareas reduce producen la salida' width='640'></p>

Con este modelo en mente, el resto de la sección ejecuta WordCount tal cual
lo haría cualquier trabajo MapReduce: preparar la entrada, enviar el trabajo
a YARN y leer la salida.

### Ejemplo: el mismo WordCount escrito con MrJob

El ejemplo `wordcount` que se ejecuta más abajo contra YARN está escrito en
Java. Para que la correspondencia entre la teoría y el código se vea de un
vistazo, esta es la misma cuenta de palabras escrita en Python con
[**MrJob**](https://mrjob.readthedocs.io/), una biblioteca que expone
directamente las funciones `mapper` y `reducer` y oculta el resto del motor
MapReduce (*shuffle*, combinador, reintentos...). El curso ya no usa MrJob en
las prácticas —se trabaja con Spark—, pero verlo una vez ayuda a fijar el
modelo:

```python
from mrjob.job import MRJob


class MRWordCount(MRJob):
    def mapper(self, _, line: str):
        # map(K1, V1) -> list(K2, V2)
        # K1 (el número de línea) no se usa; V1 es la línea de texto.
        # Cada palabra de la línea produce un par intermedio (palabra, 1).
        for word in line.split():
            yield word.lower(), 1

    def combiner(self, word: str, counts: list[int]):
        # Suma parcial en el mismo nodo que ejecutó el mapper, antes de
        # enviar nada por la red. Válido porque sumar es conmutativo y
        # asociativo (ver más arriba).
        yield word, sum(counts)

    def reducer(self, word: str, counts: list[int]):
        # reduce(K2, list(V2)) -> (K3, list(V3))
        # `counts` es list(V2): los valores que MrJob ya agrupó por clave
        # durante el shuffle. Aquí la lista de salida tiene un único valor.
        yield word, sum(counts)


if __name__ == "__main__":
    MRWordCount.run()
```

Guardado como `wordcount_mrjob.py`, se ejecuta en local sin Hadoop ni
clúster —MrJob agrupa las claves como haría el *shuffle* antes de llamar al
reducer—. Usando el mismo contenido de `s1-a.txt`, `s1-b.txt` y `s1-c.txt`
que se sube a HDFS más abajo:

```bash
$ printf 'hadoop hdfs mapreduce\nhadoop yarn\nhdfs hadoop\nyarn recursos\nmapreduce datos hadoop\ndatos datos\n' \
    | python3 wordcount_mrjob.py
```

```text
"yarn"	2
"recursos"	1
"mapreduce"	2
"hdfs"	2
"hadoop"	4
"datos"	3
```

MrJob serializa cada par intermedio como JSON —de ahí las comillas en las
palabras—. El orden entre palabras distintas no está garantizado (aquí no
coincide con el alfabético, a diferencia de la salida de Hadoop más abajo,
que sí usa un único reducer y por eso queda ordenada); lo que importa es
que el recuento de cada palabra es el mismo que va a producir el
`wordcount` de Hadoop sobre estos mismos ficheros. Comparar ambas salidas
permite comprobar que `mapper`/`reducer` en MrJob y `map`/`reduce` en el
`wordcount` de Hadoop calculan exactamente lo mismo; lo único que cambia es
qué motor reparte el trabajo entre las máquinas.

In [ ]:
%%diapositiva pregunta
# Pregunta guía
- ¿En qué contenedor se ejecuta la orden `yarn jar ...`?
- ¿Ese contenedor es el mismo que ejecuta las tareas `map` y `reduce`?
- ¿Cómo se puede comprobar, después de lanzar el trabajo, dónde se ejecutó cada tarea?

## Ejecutar un trabajo MapReduce con YARN

Se utiliza el ejemplo `wordcount` incluido en la instalación de Hadoop, que
cuenta cuántas veces aparece cada palabra. Aunque el programa es pequeño,
recorre el mismo camino básico que una aplicación MapReduce real: el cliente
prepara y envía la aplicación a YARN, el ResourceManager asigna recursos para
el ApplicationMaster, se crean tareas `map` que leen las entradas, los
resultados intermedios se agrupan por clave durante el *shuffle*, y una o más
tareas `reduce` producen los ficheros finales en HDFS. El JAR de ejemplos ya
forma parte de la imagen; el patrón `hadoop-mapreduce-examples-*.jar` evita
repetir el número de versión en la orden.

MapReduce exige que el directorio de salida **no exista**, para evitar
sobrescribir resultados por accidente. Antes de (re)ejecutar el trabajo se
borra sólo la salida anterior; los ficheros de entrada no se tocan.

In [ ]:
!docker exec namenode su - luser -c 'hdfs dfs -rm -r -f /user/luser/s1/output'

Esta orden se lanza en `namenode`, pero eso no significa que el NameNode
ejecute las tareas: el proceso que arranca allí es el **cliente**. YARN
asigna los contenedores de ejecución a los NodeManagers de los `datanodeN`.
Durante la ejecución se muestran un identificador de aplicación y los
porcentajes de avance de las fases `map` y `reduce`; el trabajo debe terminar
con éxito. Si falla, conserva el identificador y revisa primero el mensaje de
error — borrar todo el clúster debería ser el último recurso, no la primera
respuesta.

In [ ]:
!docker exec namenode su - luser -c 'yarn jar "$HADOOP_HOME"/share/hadoop/mapreduce/hadoop-mapreduce-examples-*.jar wordcount /user/luser/s1/input /user/luser/s1/output'

Debe aparecer `_SUCCESS` (finalización correcta) y al menos un fichero `part-r-00000` (el sufijo `r` indica que es salida de una tarea `reduce`).

In [ ]:
!docker exec namenode su - luser -c 'hdfs dfs -ls /user/luser/s1/output'

Se esperan estos recuentos (Hadoop separa palabra y contador con un
tabulador, así que el espaciado exacto puede variar):

```text
datos      3
hadoop     4
hdfs       2
mapreduce  2
recursos   1
yarn       2
```

In [ ]:
!docker exec namenode su - luser -c 'hdfs dfs -cat /user/luser/s1/output/part-r-*'

### Consultar la aplicación en YARN y en el Timeline Server

`mapred job -list all` presenta con la nomenclatura de MapReduce las
aplicaciones YARN de tipo MapReduce: consulta al ResourceManager, así que no
necesita un JobHistoryServer (este clúster no lo arranca, como se explicó al
observar el Timeline Server). Después de ejecutar `wordcount` debe aparecer con
un identificador `job_...` —con los mismos números que el `application_...` de
YARN— y estado `SUCCEEDED`.

In [ ]:
!docker exec namenode su - hdadmin -c 'mapred job -list all'

`yarn application -list -appStates ALL` muestra el mismo trabajo desde la perspectiva de YARN. Busca un estado `FINISHED` y un resultado final `SUCCEEDED`; la interfaz web de <http://localhost:8088> muestra la misma información de forma gráfica.

In [ ]:
!docker exec namenode su - hdadmin -c 'yarn application -list -appStates ALL'

Ahora que la aplicación ya no está activa, es un buen momento para localizarla
en la interfaz de histórico del Timeline Server
(<http://localhost:8188/applicationhistory/>) y comparar su estado con el
mostrado por el ResourceManager. La distribución exacta de tareas entre
DataNodes puede cambiar entre ejecuciones: un planificador distribuido no
promete que cada trabajo pequeño utilice todos los nodos por igual. Lo que
importa comprobar aquí es que el cliente entrega la aplicación a YARN, que
los NodeManagers aportan contenedores y que el resultado se escribe en HDFS.

In [ ]:
%%diapositiva resumen
# Recapitulación de la sesión 1
- HDFS separa metadatos (NameNode) de bloques (DataNodes) y los replica
- YARN separa la gestión de recursos (ResourceManager/NodeManager) de la aplicación (ApplicationMaster)
- MapReduce ejecuta map -> shuffle -> reduce sobre los datos, no al revés
- `stop` y `down` conservan los datos HDFS (viven en volúmenes); sólo borrar los volúmenes devuelve el HDFS al estado de las imágenes
> Siguiente sesión: los mismos datos, pero como ficheros Parquet en `/datalake/raw/tpcds`.

## Detener, limpiar y recuperar el entorno

No ejecutes estas órdenes por accidente mientras trabajas: se presentan como
texto para copiarlas a una terminal cuando de verdad las necesites, no como
celdas de código, precisamente para que un «ejecutar todo» del notebook no
destruya tu propio trabajo.

| Situación | Orden | Consecuencia |
| --- | --- | --- |
| Parar temporalmente | `docker compose -f ../entorno/compose-hadoop-cluster.yml stop` | Conserva contenedores y datos HDFS; se reanuda con `start`. Tras `start` los demonios tardan unos segundos en registrarse: repite `dfsadmin -report` y `yarn node -list -all` antes de continuar. |
| Reanudar | `docker compose -f ../entorno/compose-hadoop-cluster.yml start` | Vuelve a poner en marcha los mismos contenedores. `up -d` también sirve: Compose reutiliza los contenedores existentes mientras su configuración no obligue a recrearlos. |
| Limpiar sólo tus datos de la sesión | `docker exec namenode su - luser -c 'hdfs dfs -rm -r -f /user/luser/s1'` | Borra el área `/user/luser/s1` para repetir la sesión desde cero sin regenerar el clúster. No borra `/warehouse`, `/tmp`, `/user` ni los directorios de otros usuarios, ni detiene contenedores ni borra `/tmp` local. |
| Eliminar los contenedores | `docker compose -f ../entorno/compose-hadoop-cluster.yml down` | Elimina los contenedores, pero **conserva los datos HDFS**: viven en los volúmenes con nombre `namenode-data`/`datanodeN-data`, no en la capa de escritura de los contenedores. Sí borra el almacén local (no persistido) del Timeline Server. La red externa `hadoop-cluster` no se elimina. |
| Recrear los contenedores | `down` seguido de `up -d` | Los mismos volúmenes HDFS se vuelven a montar: `/user/luser`, `/datalake`, `/warehouse`, los datos de `/user/luser/s1` y un TPC-DS ya materializado siguen ahí. No hace falta reconstruir las imágenes. |
| Borrar también el HDFS (destructivo) | `docker volume rm tcdm-26-27-namenode-data tcdm-26-27-datanode1-data tcdm-26-27-datanode2-data tcdm-26-27-datanode3-data` (con los contenedores ya eliminados) | Vacía el espacio de nombres y los bloques: el próximo `up -d` arranca con un HDFS recién formateado, como la primera vez. Es la única orden de esta tabla que borra `/datalake/raw/tpcds` y cualquier otro dato ya materializado en sesiones posteriores. A partir de la Sesión 2, `entorno/Makefile` envuelve esta misma orden como `make clean-hdfs`. |

### Diagnosticar un arranque incompleto

Si algo no arranca como se espera, conviene mirar en este orden: primero qué
contenedores siguen activos (`docker compose ... ps`), después las últimas
líneas de todos los servicios (`docker compose ... logs --tail=120`) y, si el
problema parece localizado, los logs de un contenedor concreto (`docker logs
namenode --tail=120`, `docker logs datanode1 --tail=120`). Los fallos más
habituales en esta fase son que Docker no tenga recursos suficientes, que la
red externa no exista todavía, o que los demonios sigan arrancando. Después
de revisar los logs, repite las comprobaciones de HDFS y YARN de este
notebook: no hace falta reconstruir las imágenes para una incidencia
habitual.

## Preguntas para interpretar la experiencia

Antes de dar por terminada la sesión, conviene poder responder con las
propias palabras:

- ¿Qué diferencia hay entre que Docker muestre un contenedor como `running` y
  que Hadoop muestre un DataNode como `Live`?
- ¿Por qué `ls /tmp` (en el contenedor) y `hdfs dfs -ls /tmp` no consultan el
  mismo lugar?
- ¿Qué información guarda el NameNode y qué información guardan los
  DataNodes?
- ¿Por qué tres réplicas no aparecen como tres ficheros al ejecutar
  `hdfs dfs -ls`?
- ¿Qué diferencia existe entre un DataNode y un NodeManager, aunque ambos se
  ejecuten dentro del mismo contenedor?
- ¿Por qué MapReduce rechaza un directorio de salida que ya existe?
- ¿Qué parte de la orden `yarn jar` actúa como cliente y dónde se ejecutan
  realmente las tareas?
- ¿Qué diferencia hay entre la vista de aplicaciones activas del
  ResourceManager y el histórico del Timeline Server?
- ¿Por qué `timelineserver` resuelve al mismo contenedor que `namenode` y qué
  ventaja didáctica tiene conservar el nombre de servicio separado?
- ¿Por qué los datos HDFS sobreviven a `stop` y también a `down`, y qué hay que
  borrar para devolver el laboratorio al estado inicial de las imágenes?

## Evidencias para la siguiente sesión

Antes de la segunda o tercera sesión tendrás una reunión individual breve
(unos minutos) con el profesor para revisar el trabajo de esta sesión. Esa
reunión combina dos partes distintas: una demostración en vivo sobre tu
propio ordenador y una memoria escrita breve. Mantén el entorno de esta
sesión operativo hasta entonces.

### Qué mostrar en el ordenador durante la reunión

Ten preparado y a mano, funcionando en tu propio equipo:

1. `docker compose ... ps` con los cuatro contenedores en ejecución.
2. `hdfs dfsadmin -report` con tres DataNodes vivos.
3. `yarn node -list -all` con tres NodeManagers en estado `RUNNING`.
4. La interfaz de histórico del Timeline Server en
   <http://localhost:8188/applicationhistory/>, con una aplicación YARN
   visible.
5. Los ficheros copiados en `/user/luser/s1/input` y las ubicaciones de
   bloques obtenidas con `hdfs fsck`.
6. El resultado correcto de WordCount en `/user/luser/s1/output`
   (`hadoop 4`, `mapreduce 2`, `datos 3`, `hdfs 2`, `yarn 2`, `recursos 1`).
7. Que sabes detener y regenerar el clúster sin reconstruir las imágenes.

### Memoria escrita (una o dos páginas)

Trae también un documento breve —una o dos páginas, no hace falta más— que
no se limite a pegar capturas de las órdenes anteriores: debe explicar con
tus propias palabras la relación entre NameNode y DataNode, la relación
entre ResourceManager y NodeManager, la diferencia entre una ruta local y
una ruta HDFS, y qué ocurre con los datos al usar `stop` frente a `down`.

A partir de ahora conserva este entorno: las siguientes sesiones usarán el
mismo HDFS para acceder a datos desde Arrow, Spark y las herramientas de
catálogo.
